<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/onlineshopping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Compulsary RUN

install command

In [ ]:
%%capture
!pip install polars xlsxwriter fastexcel
!pip install playwright beautifulsoup4 polars
!playwright install
!pip install itables
!playwright install-deps
!pip install curl_cffi

import lib

In [ ]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import xlsxwriter
import datetime
from datetime import date

## Lotus's

In [ ]:
###
# --- This code work
###


async def scrape_lotuss_scroller(shop_url: str):
    extracted_data = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        print(f"Opening shop... Please wait.")
        await page.goto(shop_url, wait_until="domcontentloaded")

        try:
            # Wait for the first batch of products to load
            await page.wait_for_selector(".mui-style-17swnep", timeout=15000)
        except Exception as e:
            print("Failed to load the initial products.")
            await browser.close()
            return pl.DataFrame()

        # --- THE INFINITE SCROLLER ---
        previous_item_count = 0
        scroll_attempts = 0
        max_scrolls = 30 # Safety net so it doesn't scroll literally forever

        while scroll_attempts < max_scrolls:
            # Tell the browser to scroll to the absolute bottom of the page
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")

            # Wait 4 seconds for Lotus's servers to fetch and draw the next batch of products
            await asyncio.sleep(4)

            # Count how many products are currently physically on the screen
            current_items = await page.query_selector_all(".MuiCard-root")
            current_count = len(current_items)

            print(f"Scroll {scroll_attempts + 1}: Found {current_count} products on screen...")

            # If we scrolled down but the number of products didn't increase, we hit the bottom!
            if current_count == previous_item_count:
                print("Hit the bottom of the catalog! Scraping all loaded data...")
                break

            previous_item_count = current_count
            scroll_attempts += 1

        # Now that the browser has fully expanded all products, take a snapshot of the HTML
        html_content = await page.content()
        soup = BeautifulSoup(html_content, "html.parser")
        products = soup.find_all("div", class_="MuiCard-root")

        for prod in products:
            # 1. Extract Name
            name_elem = prod.find(class_="mui-style-17swnep")
            name = name_elem.text.strip() if name_elem else None

            if not name:
                continue

            # 2. Extract Promotion Price
            promo_elem = prod.find("div", class_="mui-style-18s6ztp")
            if promo_elem:
                raw_promo = promo_elem.text.replace(',', '').replace('฿', '').strip()
                promotion_price = raw_promo if raw_promo else None
            else:
                promotion_price = None

            # 3. Extract Original Price
            orig_elem = prod.find("div", class_="mui-style-f59hd5")
            if orig_elem:
                raw_orig = orig_elem.text.replace(',', '').replace('฿', '').strip()
                original_price = raw_orig if raw_orig else promotion_price
            else:
                original_price = promotion_price

            extracted_data.append({
                "name": name,
                "promotion_price": promotion_price,
                "original_price": original_price
            })

        await browser.close()

    df = pl.DataFrame(extracted_data)

    # Final cleanup of any accidental duplicates
    if not df.is_empty():
        df = df.unique(subset=["name"], maintain_order=True)

    return df

In [ ]:
# --- RUN IT ---
# Notice we are just passing the raw URL without trying to add "&page="
shop_laundry_supplies_url = "https://www.lotuss.com/en/category/household-and-merits/86590-laundry-supplies?sort=relevance:DESC&filter.brandId=21490,21430,21829,22054,23485"
df_laundry_supplies = await scrape_lotuss_scroller(shop_url=shop_laundry_supplies_url)

print("\n--- Scraping Complete ---")
print(df_laundry_supplies)


shop_dish_washing_liquid_url = "https://www.lotuss.com/en/category/household-and-merits/86590-cleaning-chemical/86671-dish-washing-liquid?sort=relevance:DESC&filter.brandId=22327"
df_lotuss_dish_washing_liquid = await scrape_lotuss_scroller(shop_url=shop_dish_washing_liquid_url)

print("\n--- Scraping Complete ---")
print(df_lotuss_dish_washing_liquid)

Opening shop... Please wait.
Scroll 1: Found 45 products on screen...
Scroll 2: Found 75 products on screen...
Scroll 3: Found 105 products on screen...
Scroll 4: Found 135 products on screen...
Scroll 5: Found 165 products on screen...
Scroll 6: Found 195 products on screen...
Scroll 7: Found 225 products on screen...
Scroll 8: Found 255 products on screen...
Scroll 9: Found 285 products on screen...
Scroll 10: Found 300 products on screen...
Scroll 11: Found 330 products on screen...
Scroll 12: Found 375 products on screen...
Scroll 13: Found 405 products on screen...
Scroll 14: Found 420 products on screen...
Scroll 15: Found 465 products on screen...
Scroll 16: Found 480 products on screen...
Scroll 17: Found 510 products on screen...
Scroll 18: Found 540 products on screen...
Scroll 19: Found 570 products on screen...
Scroll 20: Found 579 products on screen...
Scroll 21: Found 579 products on screen...
Hit the bottom of the catalog! Scraping all loaded data...

--- Scraping Comple

In [ ]:
df_lotuss = pl.concat([df_lotuss_dish_washing_liquid, df_laundry_supplies])
df_lotuss = df_lotuss.with_columns(
    # IF the promotion price is exactly the same as the original price...
    pl.when(pl.col("promotion_price") == pl.col("original_price"))
    # THEN fill the cell with a true Null (None) value...
    .then(None)
    # OTHERWISE keep the promotion price exactly as it is.
    .otherwise(pl.col("promotion_price"))
    .alias("promotion_price")
)

## Big C

In [ ]:
!pip install playwright polars beautifulsoup4
!playwright install chromium

In [ ]:
!pip install -q playwright polars beautifulsoup4 nest_asyncio playwright-stealth
!playwright install chromium --with-deps

Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

In [ ]:
%%script echo "Cell Skipped"
# @title still not work code
"""
BigC.co.th Scraper v4 — Cloudflare bypass via playwright-stealth
================================================================
BigC is protected by Cloudflare Bot Management (cdn-cgi/challenge-platform).
playwright-stealth patches the browser fingerprint so Cloudflare passes us through.

Install (run this cell FIRST in Colab):
    !pip install -q playwright polars beautifulsoup4 nest_asyncio playwright-stealth
    !playwright install chromium --with-deps

Then run this file.
"""

# ── Colab/Jupyter: patch the running event loop FIRST ────────────────────────
import nest_asyncio
nest_asyncio.apply()

from playwright.async_api import async_playwright
from playwright_stealth import stealth_async      # ← Cloudflare bypass
from bs4 import BeautifulSoup
import polars as pl
import asyncio, json, math, re

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DEBUG = True       # → False once you confirm the API URL below

# After first debug run, paste the product-API hostname here, e.g.:
#   CONFIRMED_API_HOST = "api.bigc.co.th"
CONFIRMED_API_HOST = ""
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PRICE_API_CANDIDATES = [
    "https://api.bigc.co.th",
    "https://www.bigc.co.th/api",
]

URL_LAUNDRY = (
    "https://www.bigc.co.th/en/category/laundry"
    "?brand=184%2C249%2C188%2C189%2C185"
)


# ── Helpers ───────────────────────────────────────────────────────────────────

def _to_float(val):
    try:
        return float(str(val).replace(",", "").strip())
    except (TypeError, ValueError):
        return None

def _find_nested(obj, *keys):
    for k in keys:
        if isinstance(obj, dict) and k in obj:
            obj = obj[k]
        else:
            return None
    return obj

def _clean_headers(raw: dict) -> dict:
    skip = {":method", ":path", ":scheme", ":authority",
            "content-length", "host", "origin", "referer"}
    return {k: v for k, v in raw.items() if k.lower() not in skip}

def extract_hits(body) -> list:
    if not isinstance(body, dict):
        return []
    if "results" in body:
        for r in body["results"]:
            if isinstance(r, dict) and r.get("hits"):
                return r["hits"]
    if "hits" in body:
        return body["hits"]
    for path in [("data","products","items"),
                 ("data","categoryProducts","items"),
                 ("data","category","products","items")]:
        items = _find_nested(body, *path)
        if isinstance(items, list) and items:
            return items
    for key in ("data","products","product_list","productList","items"):
        val = body.get(key)
        if isinstance(val, list) and val:
            return val
        if isinstance(val, dict):
            inner = val.get("items") or val.get("data") or val.get("hits")
            if isinstance(inner, list) and inner:
                return inner
    return []

def sniff_total(body: dict) -> tuple:
    for r in ([body] + body.get("results", [])):
        if not isinstance(r, dict): continue
        found = (r.get("found") or r.get("nbHits") or r.get("total")
                 or r.get("total_count") or r.get("totalCount") or 0)
        pgsz  = (r.get("size") or r.get("hitsPerPage") or r.get("page_size")
                 or r.get("per_page") or 20)
        if found:
            return int(found), int(pgsz)
    pt = _find_nested(body, "data", "products", "total_count")
    if pt:
        pi = _find_nested(body,"data","products","page_info","page_size") or 20
        return int(pt), int(pi)
    return 0, 20

def is_product_response(url: str, body: dict) -> bool:
    url_l = url.lower()
    for bad in ("analytics","track","beacon","banner","event","auth",
                "login","cart","checkout","suggest","autocomplete",
                "cms","config","i18n","locale","font","widget","gtm",
                "facebook","google","clarity","challenge"):
        if bad in url_l:
            return False
    hits = extract_hits(body)
    if len(hits) < 2:
        return False
    count = 0
    for h in hits[:5]:
        doc = h.get("document", h) if isinstance(h, dict) else {}
        has_name  = any(k in doc for k in
            ("name","name_en","title","product_name","nameEn","productName"))
        has_price = any(k in doc for k in
            ("price","sale_price","salePrice","special_price",
             "promo_price","sellingPrice","retail_price"))
        if has_name and has_price:
            count += 1
    return count >= 2

def doc_to_row(doc: dict):
    if not isinstance(doc, dict): return None
    name = (doc.get("name_en") or doc.get("nameEn") or doc.get("name")
            or doc.get("title_en") or doc.get("title")
            or doc.get("product_name") or "").strip()
    if not name: return None
    brand = doc.get("brand_name") or doc.get("brandName") or doc.get("brand") or ""
    if isinstance(brand, dict):
        brand = brand.get("name_en") or brand.get("name") or ""
    sku = str(doc.get("sku") or doc.get("product_id") or doc.get("productId")
              or doc.get("id") or "")
    promo = _to_float(
        doc.get("price") or doc.get("sale_price") or doc.get("salePrice")
        or doc.get("promo_price") or doc.get("special_price")
        or doc.get("sellingPrice"))
    orig  = _to_float(
        doc.get("original_price") or doc.get("originalPrice")
        or doc.get("regular_price") or doc.get("regularPrice")
        or doc.get("compare_at_price") or doc.get("compareAtPrice")
        or doc.get("base_price") or doc.get("mrp"))
    if orig is None: orig = promo
    in_stock  = bool(doc.get("in_stock") or doc.get("inStock")
                     or doc.get("available")
                     or (doc.get("stock_status") == "in_stock")
                     or doc.get("qty", 0))
    unit_size = str(doc.get("unit_size") or doc.get("unitSize")
                    or doc.get("pack_size") or doc.get("weight")
                    or doc.get("size") or "")
    handle    = str(doc.get("handle") or doc.get("url_key")
                    or doc.get("slug") or sku)
    return dict(name=name, brand=str(brand).strip(), sku=sku,
                promotion_price=promo, original_price=orig,
                in_stock=in_stock, unit_size=unit_size, handle=handle)

def make_paged_request(captured: dict, page_num: int):
    api_url   = captured["url"]
    post_data = captured.get("post_data")
    method    = captured.get("method","GET").upper()
    if method == "POST" and post_data:
        try:
            b = json.loads(post_data)
            if "requests" in b:
                for req in b["requests"]:
                    p = req.get("params", {})
                    if isinstance(p, dict):
                        p["page"] = page_num - 1
                    elif isinstance(p, str):
                        p = re.sub(r'page=\d+', f'page={page_num-1}', p)
                        if "page=" not in p: p += f"&page={page_num-1}"
                        req["params"] = p
            elif "searches" in b:
                for s in b["searches"]: s["page"] = page_num - 1
            else:
                for key in ("page","currentPage","pageNum"):
                    if key in b: b[key] = page_num; break
                else: b["page"] = page_num
            return api_url, json.dumps(b)
        except Exception:
            return api_url, re.sub(r'"page"\s*:\s*\d+',
                                   f'"page":{page_num}', post_data)
    if "page=" in api_url:
        return re.sub(r'page=\d+', f'page={page_num}', api_url), None
    sep = "&" if "?" in api_url else "?"
    return f"{api_url}{sep}page={page_num}", None

async def fetch_original_prices(context, rows, price_base, hdrs):
    SEM = asyncio.Semaphore(5)
    async def fetch_one(row):
        if (row["original_price"] is not None
                and row["original_price"] != row["promotion_price"]
                and row["original_price"] > (row["promotion_price"] or 0)):
            return row
        sku = row["sku"]; handle = row["handle"]
        async with SEM:
            for url in [f"{price_base}/v1/products/{sku}",
                        f"{price_base}/products/{sku}",
                        f"https://www.bigc.co.th/api/products/{handle}"]:
                try:
                    r = await context.request.get(url, headers=hdrs)
                    if r.status != 200: continue
                    body = await r.json()
                    cmp = (_find_nested(body,"data","compare_at_price")
                           or _find_nested(body,"data","compareAtPrice")
                           or _find_nested(body,"compare_at_price"))
                    v = _to_float(cmp)
                    if v and v > (row["promotion_price"] or 0):
                        row["original_price"] = v; return row
                except Exception: pass
        return row
    return list(await asyncio.gather(*[fetch_one(r) for r in rows]))


# ── Main scraper ──────────────────────────────────────────────────────────────

async def scrape_bigc(base_url=URL_LAUNDRY, fetch_prices=True, wait_seconds=20):
    extracted_data = []
    captured       = {}
    extra_headers  = {}
    debug_log      = []
    capture_done   = asyncio.Event()

    page1_url = (base_url.format(1)
                 if ("{}" in base_url or "{page}" in base_url) else base_url)

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--no-sandbox",
                "--disable-setuid-sandbox",
                "--disable-dev-shm-usage",
                # Pretend to have a real GPU / display
                "--disable-gpu-sandbox",
                "--window-size=1920,1080",
            ],
        )
        ctx = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1920, "height": 1080},
            locale="en-US",
            timezone_id="Asia/Bangkok",
            extra_http_headers={
                "Accept-Language": "en-US,en;q=0.9,th;q=0.8",
                "Accept": ("text/html,application/xhtml+xml,application/xml;"
                           "q=0.9,image/avif,image/webp,*/*;q=0.8"),
                "sec-ch-ua": '"Chromium";v="124", "Google Chrome";v="124", "Not-A.Brand";v="99"',
                "sec-ch-ua-mobile": "?0",
                "sec-ch-ua-platform": '"Windows"',
            },
        )
        tab = await ctx.new_page()

        # ── Apply stealth BEFORE navigating ──────────────────────────────
        # This patches navigator.webdriver, plugins, languages, etc.
        await stealth_async(tab)
        print("  ✓ Stealth mode applied (Cloudflare bypass)")

        # ── Response / request listeners ──────────────────────────────────
        async def on_response(response):
            url    = response.url
            status = response.status
            if any(url.endswith(e) for e in
                   (".js",".css",".png",".jpg",".jpeg",".svg",
                    ".woff",".woff2",".ico",".webp",".gif")):
                return

            ct = (response.headers.get("content-type") or "").lower()

            if DEBUG:
                body_info = ""
                hits_count = 0
                if "json" in ct or "javascript" in ct:
                    try:
                        body = await response.json()
                        hits = extract_hits(body)
                        hits_count = len(hits)
                        if hits_count:
                            first = hits[0]
                            doc   = first.get("document", first) if isinstance(first, dict) else {}
                            name  = (doc.get("name") or doc.get("name_en")
                                     or doc.get("title") or "?")
                            body_info = f'  ← {hits_count} hits, e.g. "{str(name)[:40]}"'
                    except Exception:
                        pass
                elif "html" in ct:
                    body_info = "  (HTML)"

                debug_log.append((status, url[:110], ct[:35], hits_count))
                # Print anything from bigc.co.th or with hits
                if hits_count or "bigc.co.th" in url.lower():
                    print(f"  [{status}] {url[:100]}{body_info}")

            if capture_done.is_set(): return
            if status != 200: return
            if "json" not in ct and "javascript" not in ct: return

            try:
                body = await response.json()
            except Exception:
                return

            if CONFIRMED_API_HOST and CONFIRMED_API_HOST not in url:
                return
            if not is_product_response(url, body):
                return

            req = response.request
            captured.update(url=url, method=req.method,
                            headers=await req.all_headers(),
                            post_data=req.post_data)
            print(f"\n  ✓ Product API: {url[:100]}")
            capture_done.set()

        async def on_request(request):
            if "bigc.co.th" in request.url and not extra_headers:
                try:
                    h = await request.all_headers()
                    skip = {":method",":path",":scheme",":authority","content-length"}
                    extra_headers.update(
                        {k:v for k,v in h.items() if k.lower() not in skip})
                except Exception: pass

        tab.on("response", on_response)
        tab.on("request",  on_request)

        # ── Navigate ──────────────────────────────────────────────────────
        print(f"Opening BigC (with stealth)… up to {wait_seconds}s")
        try:
            await tab.goto(page1_url, wait_until="networkidle",
                           timeout=wait_seconds * 1000)
        except Exception:
            pass   # timeout on networkidle is fine — responses already captured

        # Human-like scroll
        for pct in (0.3, 0.6, 1.0):
            try:
                await tab.evaluate(
                    f"window.scrollTo(0, document.body.scrollHeight * {pct})")
                await asyncio.sleep(1.2)
            except Exception:
                pass

        await asyncio.sleep(3)

        # ── Check if Cloudflare challenge is still showing ────────────────
        page_title = await tab.title()
        if "just a moment" in page_title.lower() or "checking" in page_title.lower():
            print(f"\n  ⚠ Cloudflare challenge still active (title: '{page_title}')")
            print("    Waiting 10 more seconds for it to resolve…")
            await asyncio.sleep(10)
            # Scroll again after challenge
            for pct in (0.5, 1.0):
                try:
                    await tab.evaluate(
                        f"window.scrollTo(0, document.body.scrollHeight * {pct})")
                    await asyncio.sleep(1)
                except Exception:
                    pass
        else:
            print(f"  ✓ Page title: '{page_title[:60]}'")

        # ── DEBUG report ──────────────────────────────────────────────────
        if DEBUG:
            hits_urls = [(s, u, ct, n) for s,u,ct,n in debug_log if n > 0]

            print("\n" + "━"*72)
            print("DEBUG REPORT")
            print("━"*72)

            if hits_urls:
                print(f"\n🎯 Product-like JSON responses ({len(hits_urls)}):")
                for s, u, ct, n in hits_urls:
                    print(f"   [{s}] {n:4} hits  {u}")
                print("\n▶ ACTION:")
                print("  1. Pick the product URL above.")
                print("  2. Set  CONFIRMED_API_HOST = '<hostname>'")
                print("  3. Set  DEBUG = False  and re-run.")
            else:
                print("\n⚠  No JSON product responses detected.")
                # Show all BigC requests
                bigc_reqs = [(s,u,ct) for s,u,ct,n in debug_log
                             if "bigc" in u.lower()]
                print(f"\n   All BigC responses captured ({len(bigc_reqs)}):")
                for s,u,ct in bigc_reqs[:40]:
                    print(f"   [{s}] {ct[:30]:32} {u[:85]}")

                # SSR check
                print("\n   Checking page HTML for products (SSR)…")
                html = await tab.content()
                print(f"   Page title : '{await tab.title()}'")
                print(f"   HTML length: {len(html):,} chars")
                clues = ["product","sku","price","laundry","detergent","__NEXT_DATA__"]
                for c in clues:
                    found = c.lower() in html.lower()
                    print(f"   {'✓' if found else '✗'} '{c}' in HTML")

                # Try to pull products from __NEXT_DATA__ if present
                soup = BeautifulSoup(html, "html.parser")
                el   = soup.find("script", id="__NEXT_DATA__")
                if el:
                    try:
                        jd    = json.loads(el.string)
                        props = jd.get("props",{}).get("pageProps",{})
                        print(f"\n   __NEXT_DATA__ pageProps keys: {list(props.keys())[:15]}")
                        # Try every value for product lists
                        for k, v in props.items():
                            hits = extract_hits(v) if isinstance(v, dict) else []
                            if hits:
                                print(f"   🎯 Found {len(hits)} hits under pageProps['{k}']!")
                                doc = hits[0].get("document", hits[0]) if isinstance(hits[0],dict) else {}
                                print(f"      First product keys: {list(doc.keys())[:10]}")
                    except Exception as e:
                        print(f"   __NEXT_DATA__ parse error: {e}")
                else:
                    print("   ✗ No __NEXT_DATA__ script tag found")

            print("━"*72)
            await browser.close()
            return pl.DataFrame()

        # ── Production path ───────────────────────────────────────────────
        if not capture_done.is_set():
            print("⚠ Product API not captured.")
            await browser.close()
            return pl.DataFrame()

        # SSR page 1
        html = await tab.content()
        soup = BeautifulSoup(html, "html.parser")
        el   = soup.find("script", id="__NEXT_DATA__")
        total_found = 0; page_size = 20
        if el:
            try:
                jd    = json.loads(el.string)
                props = jd.get("props",{}).get("pageProps",{})
                sr    = (props.get("initialData") or props.get("categoryData")
                         or props.get("searchResult")
                         or props.get("initialSearchResult") or {})
                total_found, page_size = sniff_total(sr)
                for h in extract_hits(sr):
                    row = doc_to_row(h.get("document", h))
                    if row: extracted_data.append(row)
                if extracted_data:
                    print(f"  SSR page 1: {len(extracted_data)} products")
            except Exception as e:
                print(f"  ⚠ SSR error: {e}")

        if not extracted_data:
            hdrs = _clean_headers(captured["headers"])
            m    = captured["method"].upper()
            try:
                resp = (await ctx.request.post(captured["url"], headers=hdrs,
                                               data=captured["post_data"])
                        if m == "POST" and captured.get("post_data")
                        else await ctx.request.get(captured["url"], headers=hdrs))
                body = await resp.json()
                if not total_found:
                    total_found, page_size = sniff_total(body)
                for h in extract_hits(body):
                    row = doc_to_row(h.get("document", h))
                    if row: extracted_data.append(row)
                print(f"  API page 1: {len(extracted_data)} products")
            except Exception as e:
                print(f"  ⚠ API error: {e}")

        total_pages = max(1, math.ceil(total_found / page_size)) if total_found else 1
        print(f"  Total: {total_found} products, {total_pages} pages\n")

        clean_hdrs = _clean_headers(captured["headers"])
        method     = captured["method"].upper()
        streak     = 0
        for page_num in range(2, total_pages + 1):
            paged_url, paged_body = make_paged_request(captured, page_num)
            try:
                resp = (await ctx.request.post(paged_url, headers=clean_hdrs,
                                               data=paged_body)
                        if method == "POST" and paged_body
                        else await ctx.request.get(paged_url, headers=clean_hdrs))
                if resp.status != 200:
                    print(f"  Page {page_num}: HTTP {resp.status} — stop"); break
                body = await resp.json()
                hits = extract_hits(body)
            except Exception as e:
                print(f"  Page {page_num}: ERROR — {e}"); break
            if not hits:
                streak += 1
                if streak >= 2:
                    print(f"  Page {page_num}: empty — stop"); break
                continue
            streak = 0
            print(f"  Page {page_num:3}/{total_pages}: {len(hits)} products")
            for h in hits:
                row = doc_to_row(h.get("document", h))
                if row: extracted_data.append(row)

        if fetch_prices and extracted_data and extra_headers:
            print(f"\n  Enriching original prices…")
            extracted_data = await fetch_original_prices(
                ctx, extracted_data, PRICE_API_CANDIDATES[0], extra_headers)
            n = sum(1 for r in extracted_data
                    if r["original_price"] and
                    r["original_price"] != r["promotion_price"])
            print(f"  Real discounts on {n} products")

        await browser.close()

    if not extracted_data:
        print("No data collected.")
        return pl.DataFrame()

    return (
        pl.DataFrame(extracted_data)
        .drop("handle")
        .unique(subset=["sku"], maintain_order=True)
        .with_columns(
            pl.when(pl.col("original_price").is_not_null()
                    & (pl.col("original_price") > pl.col("promotion_price")))
            .then(pl.col("original_price")).otherwise(None)
            .alias("original_price"),
        )
        .with_columns(
            pl.when(pl.col("original_price").is_not_null()
                    & (pl.col("original_price") > 0))
            .then(((pl.col("original_price") - pl.col("promotion_price"))
                   / pl.col("original_price") * 100).round(1))
            .otherwise(None).alias("discount_pct"),
        )
        .sort("name")
    )


# ── Run ───────────────────────────────────────────────────────────────────────
df_bigc = await scrape_bigc()

Cell Skipped


## Makro

url_detergent =https://www.makro.pro/en/c/collections/Home%20Care%20%7C%20DETERGENT%20Fresh%20Scent/17983?info=JTdCJTIyc291cmNlRXZlbnQlMjIlM0ElMjJmbGV4aXBhZ2VDYXJvdXNlbCUyMiUyQyUyMmNhcm91c2VsTmFtZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwREVURVJHRU5UJTIwRnJlc2glMjBTY2VudCUyMiUyQyUyMmNhcm91c2VsVGl0bGUlMjIlM0ElMjJIb21lJTIwQ2FyZSUyMCU3QyUyMERFVEVSR0VOVCUyMEZyZXNoJTIwU2NlbnQlMjIlN0Q

url_fresh_soft =
https://www.makro.pro/en/c/collections/Home%20Care%20%7C%20Fresh%20and%20Soft/17985?info=JTdCJTIyc291cmNlRXZlbnQlMjIlM0ElMjJmbGV4aXBhZ2VDYXJvdXNlbCUyMiUyQyUyMmNhcm91c2VsTmFtZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRnJlc2glMjBhbmQlMjBTb2Z0JTIyJTJDJTIyY2Fyb3VzZWxUaXRsZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRnJlc2glMjBhbmQlMjBTb2Z0JTIyJTdE

<span class="css-1n8x1y2">HYGIENE Ironing Smooth Pink Floral Pink 500 ml x 3</span>

<p class="MuiTypography-root MuiTypography-body1 css-tn2bzw">27</p>
<p class="MuiTypography-root MuiTypography-body1 css-si1rsb">฿ 38.5</p>


In [ ]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import re

async def scrape_makro_spa_clicker(start_url: str):
    extracted_data = []
    seen_names = set() # Track names so we don't scrape duplicates if the page is slow

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled"]
        )
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1920, "height": 1080}
        )

        page = await context.new_page()
        print("Walking through Makro's front door...")

        # Load the base URL ONCE
        await page.goto(start_url, wait_until="networkidle")
        await asyncio.sleep(6) # Wait for initial hydration

        assume_MAX_page = 16

        for page_num in range(1, assume_MAX_page + 1):
            print(f"Scraping page {page_num}...")

            # Scroll to the bottom to force lazy-loaded images and prices to render
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await asyncio.sleep(2)

            # Take a snapshot of the DOM
            html_content = await page.content()
            soup = BeautifulSoup(html_content, "html.parser")

            # Makro wraps product cards in <a> tags linking to /p/ (product pages)
            product_cards = soup.find_all("a", href=lambda href: href and "/p/" in href)

            new_items_found = 0

            for card in product_cards:
                texts = list(card.stripped_strings)
                if len(texts) < 3:
                    continue

                # 1. Hunt for the Product Name
                name = None
                for text in texts:
                    if len(text) > 10 and "฿" not in text and "points" not in text.lower() and "Today" not in text:
                        name = text
                        break

                if not name or name in seen_names:
                    continue

                # 2. Hunt for the Prices (Hyper-Resilient Float Extractor)
                prices = []
                for text in texts:
                    if "buy" in text.lower() or "get" in text.lower() or "point" in text.lower():
                        continue

                    # Strip out currency symbols and commas
                    clean_text = text.replace("฿", "").replace(",", "").strip()

                    try:
                        val = float(clean_text)
                        # Safeguard: Ensure it's an actual price, not a quantity like "3 options"
                        if 5 < val < 10000:
                            prices.append(val)
                    except ValueError:
                        pass

                if not prices:
                    continue

                # 3. The E-Commerce Rule: Smallest is promo, Largest is original
                promo_price = min(prices)
                original_price = max(prices)

                extracted_data.append({
                    "name": name,
                    "promotion_price": promo_price,
                    "original_price": original_price
                })

                seen_names.add(name)
                new_items_found += 1

            print(f"  -> Extracted {new_items_found} new products.")

            # 4. THE SPA CLICKER
            if page_num < assume_MAX_page:
                try:
                    # Find the pagination "Next" button and click it
                    next_button = page.locator("text=Next").first

                    if await next_button.is_visible():
                        await next_button.click()
                        print("  -> Clicked 'Next'. Waiting for SPA to load...")
                        await asyncio.sleep(4) # Give React time to swap the products
                    else:
                        print("  -> 'Next' button not visible. Reached the end of the catalog!")
                        break
                except Exception as e:
                    print(f"  -> Pagination stopped: {e}")
                    break

        await browser.close()

    # --- POLARS CLEANUP ---
    df = pl.DataFrame(extracted_data)

    if not df.is_empty():
        df = df.unique(subset=["name"], maintain_order=True)

        # Nullify fake promotions
        df = df.with_columns(
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )

        # Calculate the Discount Percentage
        df = df.with_columns(
            pl.when(pl.col("original_price").is_not_null())
            .then(((pl.col("original_price") - pl.col("promotion_price")) / pl.col("original_price") * 100).round(1))
            .otherwise(None)
            .alias("discount_pct")
        ).sort("name")

    return df

# --- RUN IT ---
# Notice we DO NOT add "&page={}" to the URL anymore. We let the clicker do the work!
url_detergent = "https://www.makro.pro/en/c/collections/Home%20Care%20%7C%20DETERGENT%20Fresh%20Scent/17983?info=JTdCJTIyc291cmNlRXZlbnQlMjIlM0ElMjJmbGV4aXBhZ2VDYXJvdXNlbCUyMiUyQyUyMmNhcm91c2VsTmFtZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwREVURVJHRU5UJTIwRnJlc2glMjBTY2VudCUyMiUyQyUyMmNhcm91c2VsVGl0bGUlMjIlM0ElMjJIb21lJTIwQ2FyZSUyMCU3QyUyMERFVEVSR0VOVCUyMEZyZXNoJTIwU2NlbnQlMjIlN0Q"

url_fresh_soft = "https://www.makro.pro/en/c/collections/Home%20Care%20%7C%20Fresh%20and%20Soft/17985?info=JTdCJTIyc291cmNlRXZlbnQlMjIlM0ElMjJmbGV4aXBhZ2VDYXJvdXNlbCUyMiUyQyUyMmNhcm91c2VsTmFtZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRnJlc2glMjBhbmQlMjBTb2Z0JTIyJTJDJTIyY2Fyb3VzZWxUaXRsZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRnJlc2glMjBhbmQlMjBTb2Z0JTIyJTdE"

df_makro_detergent = await scrape_makro_spa_clicker(start_url=url_detergent)

print("\n" + "=" * 60)
print(f"Makro Scraping Complete - {len(df_makro_detergent)} unique products")
print("=" * 60)
print(df_makro_detergent)

df_makro_freshsoft = await scrape_makro_spa_clicker(start_url=url_fresh_soft)
print("\n" + "=" * 60)
print(f"Makro Scraping Complete - {len(df_makro_freshsoft)} unique products")
print("=" * 60)
print(df_makro_freshsoft)

Walking through Makro's front door...
Scraping page 1...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 2...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 3...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 4...
  -> Extracted 0 new products.
  -> 'Next' button not visible. Reached the end of the catalog!

Makro Scraping Complete - 60 unique products
shape: (60, 4)
+-----------------------------------+-----------------+----------------+--------------+
| name                              | promotion_price | original_price | discount_pct |
| ---                               | ---             | ---            | ---          |
| str                               | f64             | f64            | f64          |
+=====================================================================================+
| ALL Washing Powder Regular 3 k... | 99.0        

In [ ]:
df_makro = pl.concat([df_makro_detergent, df_makro_freshsoft])

In [ ]:
## Transform Makro
if "discount_pct" in df_makro.columns:
    df_makro = df_makro.drop("discount_pct")

In [ ]:
df_makro = df_makro.with_columns(
    # Rule 1: If original is null, fill it with the promotion price.
    original_price=pl.when(pl.col("original_price").is_null())
                     .then(pl.col("promotion_price"))
                     .otherwise(pl.col("original_price")),

    # Rule 2: If original was null, empty out the promotion price.
    promotion_price=pl.when(pl.col("original_price").is_null())
                      .then(None)
                      .otherwise(pl.col("promotion_price"))
)

Support for third party widgets will remain active for the duration of the session. To disable support:

## Transform
##### Extract Brand, Quantity, Unit

In [ ]:
# @title udf
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Define the patterns (Centralized here so you only ever update them in one place!)
    quant_unit_pattern = r"(?i)(\d+)\s*(ML|G|KG|L)\b"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(today_date).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Extract Quantity (Pro-Tip: strict=False prevents the pipeline from crashing if a weird string can't be cast to an Integer)
        pl.col("name").str.extract(quant_unit_pattern, 1).cast(pl.Int64, strict=False).alias("Quantity"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Shop")
    )



In [ ]:
# @title create pdf
# --- How to use it ---
# Instead of writing 10 lines of code per shop, you now only write one!

# 1. Transform Makro
df_makro = parse_product_names(df_makro, shop_name = "Makro")

# 2. Transform Lotus's (from yesterday's script)
df_lotuss = parse_product_names(df_lotuss, shop_name = "Lotus's")

# 3. Transform Big C (when you run it via ScrapingBee!)
# df_bigc = parse_product_names(df_bigc, shop_name = "Big C")

# And because they all have the exact same columns now, you can stack them into one massive Master Database!
# df_master = pl.concat([df_makro, df_lotuss, df_bigc])

# print(df_makro)

In [ ]:
from google.colab import data_table
data_table.disable_dataframe_formatter()

In [ ]:
print(df_makro)

shape: (260, 9)
+--------------+-------------+-------------+------------+-----+----------+------+----------+-------+
| name         | promotion_p | original_pr | Date       | ... | Quantity | Unit | Pack     | Shop  |
| ---          | rice        | ice         | ---        |     | ---      | ---  | ---      | ---   |
| str          | ---         | ---         | str        |     | i64      | str  | str      | str   |
|              | f64         | f64         |            |     |          |      |          |       |
+==================================================================================================+
| ALL Washing  | 99.0        | 143.0       | 2026-04-03 | ... | 3        | KG   | null     | Makro |
| Powder       |             |             |            |     |          |      |          |       |
| Regular 3    |             |             |            |     |          |      |          |       |
| k...         |             |             |            |     |          | 

In [ ]:
print(df_lotuss)

shape: (605, 9)
+------------+------------+------------+------------+-----+----------+------+------------+---------+
| name       | promotion_ | original_p | Date       | ... | Quantity | Unit | Pack       | Shop    |
| ---        | price      | rice       | ---        |     | ---      | ---  | ---        | ---     |
| str        | ---        | ---        | str        |     | i64      | str  | str        | str     |
|            | str        | str        |            |     |          |      |            |         |
+==================================================================================================+
| LIPON F    | null       | 165.00     | 2026-04-03 | ... | 3200     | ML   | null       | Lotus's |
| DISHWASH   |            |            |            |     |          |      |            |         |
| 3200 ML    |            |            |            |     |          |      |            |         |
| LIPON-F    | null       | 165.00     | 2026-04-03 | ... | 3200     | ML  

## Saving file

In [ ]:
import datetime
today_date = datetime.datetime.now().strftime("%Y-%m-%d")

In [ ]:
%%script echo "Cell Skipped"
# @title old code
# Transform column
quant_unit_pattern = r"(?i)(\d+)\s*(ML|G|KG|L)\b"
pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

df_lotuss = df_lotuss.with_columns(
    # Add Date to column
    pl.lit(today_date).alias('Date'),
    # Extract the brand by splitting the name by space and taking the first element
    pl.col("name").str.split(" ").list.first().alias("Brand"),
    # Extract Group 1 (the digits) and instantly convert it into a true Integer number
pl.col("name").str.extract(quant_unit_pattern, 1).cast(pl.Int64).alias("Quantity"),
# Extract Group 2 (the letters) and force them all to UPPERCASE so "ml" becomes "ML"
pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),
    pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),
    pl.lit("Lotus's").alias("Shop")
)

Cell Skipped


In [ ]:
# show(df_lotuss.select(pl.col("Pack").unique()))

In [ ]:
# @title dv
# brand_TH_EN = {
#     # "บรีส" : "BREEZE"
#     # , "ซันไลต์" : "SUNLIGHT"
#     # ,"ไฮยีน" :"HYGIENE"
#     # , : "DOWNY"
#     # , : "OMO"
#     # , : "COMFORT"
#     # , : "FINELINE"
# }

In [ ]:

lotuss_sel_name = [
    "FINELINE LIQUID PLUS GOLD 550 ML.",
    "FINELINE LIQUID PLUS GOLD 1250 ML.",
    "HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 600 ML.",
    "HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 1400 ML.",
    "PAO WIN WASH LIQUID REFILL 620 ML",
    "PAO WIN WASH LIQUID 1300 ML",
    "PAO SUPER WHITE POWDER DETERGENT1800G.P2",
    "PAO SUPER WHITE POWDER DETERGENT 2400 G",
    "ATTACK EASY POWDER HAPPY SWEET 1800G. TWIN PACK",
    "ATTACK EASY HAPPY SWEET DETERGENT 2500G.",
    "PRO POWER DETERGENT BLUE PLUS 1700 G. PACK 1+1",
    "PRO POWDER DETERGENT BLUE PLUS 2400 G.",
    "HYGIENE FABRIC SOFTENER EXPERT CARE MILKY TOUCH 480 ML.",
    "HYGIENE FABRIC SOFTENER EXPERT CARE MILKY WHITE 480 ML 2+1",
    "HYGIENE FABRIC SOFT EXPERTCARE MILKY WHITE 1000 ML",
    "HYGIENE FABRIC SOFTENER EXPERTCARE MILKY TOUCH 1000 ML 1+1",
    "LIPON F DISHWASHING HYGIENE 500 ML PACK 3",
    "LIPON F DISH WASHER XTRA HYGENIC 750 ML. PACK 2",
    "LIPON-F DISHWASH BERGAMOT GALLON 3200 ML"
]

In [ ]:
# @title write Excel

today_date = datetime.datetime.now().strftime("%Y-%m-%d")
df_lotuss.write_excel(f"lotus_fabric_{today_date}.xlsx")
df_makro.write_excel(f"makro_detergent_{today_date}.xlsx")

In [ ]:
display(df_lotuss)

Loading ITables v2.7.3 from the internet... (need help?)
